In [ ]:
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import polars as pl

from esrpy import load_lmps

# Project paths
DATA_DIR = Path.cwd() / "data" / "inputs"

# Load the full dataset
indicative_lmps, actual_lmps = load_lmps(
    actual_csv_path=DATA_DIR / "2025-10-01_to_2025-10-08_RHESS2_ESS1_actual_lmps.csv",
    forecast_csv_path=DATA_DIR / "2025-10-01_to_2025-10-08_RHESS2_ESS1_indicative_lmps.csv",
)

print(f"Loaded {len(actual_lmps)} actual LMPs")
print(f"Loaded {len(indicative_lmps)} indicative LMPs")

# Filter to a small time range for test fixtures
CPT = ZoneInfo("America/Chicago")
start_time_cpt = datetime(2025, 10, 1, 0, 0, 0, tzinfo=CPT)
end_time_cpt = datetime(2025, 10, 1, 1, 0, 0, tzinfo=CPT)

filtered_actual = (
    actual_lmps
    .filter(
        (pl.col("interval_end_cpt") >= start_time_cpt) &
        (pl.col("interval_end_cpt") <= end_time_cpt)
    )
)

filtered_indicative = (
    indicative_lmps
    .filter(
        (pl.col("interval_end_cpt") >= start_time_cpt) &
        (pl.col("interval_end_cpt") <= end_time_cpt)
    )
)

print(f"Filtered to {len(filtered_actual)} actual LMPs")
print(f"Filtered to {len(filtered_indicative)} indicative LMPs")

Loaded 2011 actual LMPs
Loaded 22121 indicative LMPs
Filtered to 12 actual LMPs
Filtered to 77 indicative LMPs


In [2]:
# Write test fixtures (matching raw CSV format from data_retrieval.ipynb)
# Remove timezone info for CSV (tests will add it back when loading)
tests_dir = Path.cwd().parent / "tests"

# For actuals, we need interval_start_cpt too - derive it from interval_end_cpt
filtered_actual.with_columns(
    [
        (pl.col("interval_end_cpt") - pl.duration(minutes=5)).dt.replace_time_zone(None).alias("interval_start_cpt"),
        pl.col("interval_end_cpt").dt.replace_time_zone(None),
    ]
).select(["interval_start_cpt", "interval_end_cpt", "lmp"]).write_csv(
    tests_dir / "test_actual_lmps.csv", datetime_format="%Y-%m-%d %H:%M:%S"
)

# For indicatives, derive interval_start_cpt from interval_end_cpt
filtered_indicative.with_columns(
    [
        pl.col("as_of_cpt").dt.replace_time_zone(None),
        (pl.col("interval_end_cpt") - pl.duration(minutes=5)).dt.replace_time_zone(None).alias("interval_start_cpt"),
        pl.col("interval_end_cpt").dt.replace_time_zone(None),
    ]
).select(["as_of_cpt", "interval_start_cpt", "interval_end_cpt", "horizon_5min_idx", "lmp"]).write_csv(
    tests_dir / "test_indicative_lmps.csv", datetime_format="%Y-%m-%d %H:%M:%S"
)